In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import predict

In [2]:
# 1. Two example transactions
normal = {"amount_inr": 250, "hour_of_day": 14, "transaction_type": "P2M",
          "merchant_category": "Food", "transaction_status": "SUCCESS",
          "sender_age_group": "26-35", "receiver_age_group": "26-35",
          "sender_state": "Maharashtra", "sender_bank": "HDFC", "receiver_bank": "HDFC",
          "device_type": "Android", "network_type": "5G", "day_of_week": "Tuesday"}

risky = dict(normal, amount_inr=40000, hour_of_day=2, transaction_type="P2P",
             merchant_category="Other", receiver_age_group="56+", receiver_bank="SBI",
             network_type="3G", day_of_week="Sunday")

print("normal:", predict.predict_one(normal))
print("risky :", predict.predict_one(risky))

c:\Users\praja\anaconda3\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\praja\anaconda3\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\praja\anaconda3\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.1 when using version 1.9.0. This might lead to brea

normal: {'fraud_probability': 0.31879832625324434, 'prediction': 0, 'risk_level': 'Low'}
risky : {'fraud_probability': 0.4442571632608743, 'prediction': 0, 'risk_level': 'Medium'}


In [3]:
# 2. Many rows at once
df = pd.read_csv("../../data/processed/cleaned_data.csv").sample(200, random_state=1)
out = predict.predict_many(df)

display(out[["fraud_flag", "fraud_probability", "prediction", "risk_level"]].head())
print(out["risk_level"].value_counts())
print("All probabilities between 0 and 1:", out["fraud_probability"].between(0, 1).all())

,fraud_flag,fraud_probability,prediction,risk_level
240208,0,0.549806,1,High
18744,0,0.388213,0,Low
207175,0,0.435850,0,Medium
18669,0,0.654435,1,High
189086,0,0.420269,0,Medium


risk_level
High      84
Medium    82
Low       34
Name: count, dtype: int64
All probabilities between 0 and 1: True


In [4]:
# 3. Edge cases
# (a) a category the model has never seen
print("Unknown category:", predict.predict_one(dict(normal, merchant_category="NewCategory")))

# (b) very small and very big amounts
print("Amount 1     :", predict.predict_one(dict(normal, amount_inr=1)))
print("Amount 100000:", predict.predict_one(dict(normal, amount_inr=100000)))

# (c) missing columns should give a clear error
try:
    predict.predict_one({"amount_inr": 100})
except ValueError as error:
    print("Error (expected):", error)

Unknown category: {'fraud_probability': 0.31879832625324434, 'prediction': 0, 'risk_level': 'Low'}
Amount 1     : {'fraud_probability': 0.31899163026384425, 'prediction': 0, 'risk_level': 'Low'}
Amount 100000: {'fraud_probability': 0.3510714332396725, 'prediction': 0, 'risk_level': 'Low'}
Error (expected): Missing columns: hour_of_day, transaction_type, merchant_category, transaction_status, sender_age_group, receiver_age_group, sender_state, sender_bank, receiver_bank, device_type, network_type, day_of_week, is_weekend
